# Music Genre Classification - clean

In [ ]:
import os
from pathlib import Path
import shutil

import pandas as pd

import librosa
import pydub
import soundfile as sf

### Set constants, check for required dirs

In [ ]:
# Confirm that DATA_ROOT dir is readable
DATA_ROOT = Path.joinpath(Path(os.getcwd()), '../Data_Music')
print('DATA_ROOT dir exists:', DATA_ROOT.exists())

# Confirm that GENRES_DIR is readable
GENRES_DIR = DATA_ROOT / 'genres_original'
print('GENRES_DIR dir exists:', GENRES_DIR.exists())

# Print found genres
GENRES = sorted([d.name for d in GENRES_DIR.iterdir() if d.is_dir()])
print('Genres found:', GENRES)

### Make dir for processed audio files

In [ ]:
# Confirm that PROCESSED_DIR is readable
PROCESSED_DIR = DATA_ROOT / 'processed'
print('PROCESSED_DIR dir exists:', PROCESSED_DIR.exists())

for g in GENRES:
    subdir = PROCESSED_DIR / g
    if not os.path.exists(subdir):
        os.makedirs(subdir)

### Define utils for finding audio files and checking metadata

In [ ]:
def find_audio_files_within_dir(dir):
    # Traverse genre dirs, find names of each audio file
    found_audio_files = []
    for g in GENRES:
        for f in sorted((dir / g).glob('*.wav')):
            found_audio_files.append({'genre': g, 'filename': f.name, 'path': str(f)})

    # Convert found_audio_files to a pandas dataframe
    return pd.DataFrame(found_audio_files)

In [ ]:
def check_for_metadata_or_corrupt(audio_files_df):
    # Create empty list to store metadata of good files
    metadata = []
    # Create empty list to store the names and error with bad files
    corrupted = []

    # Loop through every found audio file
    for audio_file in audio_files_df.itertuples(index=False):
        try:
            # Read only the header of the file (fast, not full audio decode)
            info = sf.info(audio_file.path)
            
            # Store the metadata
            metadata.append({
                'genre': audio_file.genre,
                'filename': audio_file.filename,
                'path': audio_file.path,
                'duration': info.frames / info.samplerate,
                'sample_rate': info.samplerate,
                'channels': info.channels,
            })
        except Exception as e:
            # If the file fails store it as corrupted
            corrupted.append({
                'genre': audio_file.genre,
                'filename': audio_file.filename,
                'error': str(e)
            })

    # Convert metadata lists to dataframes
    metadata_df = pd.DataFrame(metadata)
    corrupted_df = pd.DataFrame(corrupted)

    return metadata_df, corrupted_df


### Gather info about original audio files

In [ ]:
original_audio_metadata_df, original_audio_corrupted_df = check_for_metadata_or_corrupt(
    find_audio_files_within_dir(GENRES_DIR))

print(f"Readable files: {len(original_audio_metadata_df)}")
print(f"Corrupted files: {len(original_audio_corrupted_df)}")
print(f"\nSample rate values:\n{original_audio_metadata_df['sample_rate'].value_counts()}")
print(f"\nDuration stats:\n{original_audio_metadata_df['duration'].describe()}")

### Pad, truncate, or copy files (as needed) with `pydub` library

In [ ]:
desired_duration_ms = 30000

# Loop over metadata
for audio_file in original_audio_metadata_df.itertuples():
    # Open audio file with pydub
    sound_in = pydub.AudioSegment.from_wav(audio_file.path)

    if len(sound_in) != desired_duration_ms:
        # Cut file if it's too long
        if len(sound_in) > desired_duration_ms:
            sound_out = sound_in[:desired_duration_ms]
        
        # Pad file with silence if it's too short
        else:
            # Adapted from this StackOverflow answer: https://stackoverflow.com/a/63989532
            silence = pydub.AudioSegment.silent(duration=desired_duration_ms-len(sound_in)+1)
            sound_out = sound_in + silence
        
        # Export the new sound file
        sound_out_path = PROCESSED_DIR / audio_file.genre / audio_file.filename
        sound_out.export(sound_out_path, format='wav')

    # Make a copy of file that didn't need changes and put it in
    # the `PROCESSED_DIR`
    else:
        shutil.copy(audio_file.path, PROCESSED_DIR / audio_file.genre)
    


### Interpret results from previous step about duration correcting

The duration stats below show an improvement in consistency, but it's not perfect. Maybe `pydub` applies rounding to durations, whereas `soundfile` does not.

In [ ]:
processed_audio_metadata_df, processed_audio_corrupted_df = check_for_metadata_or_corrupt(
    find_audio_files_within_dir(PROCESSED_DIR))

print(f"Readable files: {len(processed_audio_metadata_df)}")
print(f"Corrupted files: {len(processed_audio_corrupted_df)}")
print(f"\nSample rate values:\n{processed_audio_metadata_df['sample_rate'].value_counts()}")
print(f"\nDuration stats:\n{processed_audio_metadata_df['duration'].describe()}")

### Fully load audio files

In [ ]:
# Load the numpy array data for each a given audio file
def load_audio(path, sr=22050, mono=True):
    y, _ = librosa.load(path, sr=sr, mono=mono)
    return y

In [ ]:
fully_processed_df = processed_audio_metadata_df.copy()
fully_processed_df['audio'] = fully_processed_df['path'].apply(load_audio)
fully_processed_df

### Store fully processed data

In [ ]:
fully_processed_df.to_csv(PROCESSED_DIR / 'data.csv', index=False)